# Country Black Marble Extraction

Use this notebook as a template to extract NASA Black Marble nighttime lights for any country or area of interest. Start by editing the country settings below, then run the setup and extraction cells.

The notebook can:

* Download Black Marble rasters for your country boundary.
* Extract nighttime lights statistics for one or more administrative boundary files.
* Optionally extract statistics inside and outside gas-flaring buffers.
* Optionally extract statistics around border crossing or point-of-interest buffers.

## Imports

Load packages for Black Marble retrieval, geospatial processing, raster I/O, and zonal statistics.

In [ ]:
import os
from pathlib import Path
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed

import geopandas as gpd
import numpy as np
import pandas as pd
import rioxarray
from dotenv import dotenv_values
from rasterstats import zonal_stats
from shapely.geometry import box

from blackmarble import BlackMarble, Product, raster, extract

## Country Settings

Edit this cell for your country. Paths are relative to the repository `data/` folder unless you provide absolute paths.

In [ ]:
COUNTRY_NAME = "Your Country"
COUNTRY_CODE = "country"  # Short lowercase code used in output filenames, e.g. "nga" or "pak".

# Administrative boundary files to process. Add or remove levels as needed.
# Each file must contain polygon geometries.
ADMIN_BOUNDARY_FILES = {
    0: "gadm/nga_adm0.geojson",
    # 1: "gadm/nga_adm1.geojson",
    # 2: "gadm/nga_adm2.geojson",
}

# Use a projected CRS in meters for buffer creation.
# Pick a UTM/local CRS appropriate for your country.
BUFFER_CRS = "EPSG:32632"

# Date ranges. Daily/monthly products use END_DATE_DAILY_MONTHLY.
START_DATE = "2024-01-01"
END_DATE_DAILY_MONTHLY = "2024-03-31"
END_DATE_ANNUAL = "2024-01-01"

# Choose products to run. Common options:
# Product.VNP46A2 = daily, Product.VNP46A3 = monthly, Product.VNP46A4 = annual.
PRODUCTS_TO_RUN = [Product.VNP46A3]

# Optional: choose a variable. Leave as None to use the package default.
# For VNP46A2 daily gap-filled lights, use "Gap_Filled_DNB_BRDF-Corrected_NTL".
BLACKMARBLE_VARIABLE = None

# Output folders.
NTL_DIR = Path("ntl_blackmarble") / COUNTRY_CODE
RAW_SUBDIR = "raw"
AGGREGATED_SUBDIR = "aggregated"

## Optional Point Buffers

Use this for border crossings, ports, clinics, markets, or any other point locations. Your file should contain longitude and latitude columns.

In [ ]:
EXTRACT_POINT_BUFFERS = False

POINT_FILE = "border_crossings/lac_border_crossings.xlsx"
POINT_LONGITUDE_COL = "Longitude"
POINT_LATITUDE_COL = "Latitude"
POINT_ID_COL = None       # Example: "uid". Leave as None to create an ID.
POINT_NAME_COL = "Name"  # Leave as None if there is no name column.
POINT_GROUP_COL = "Border"  # Optional grouping column, e.g. neighboring country. Leave as None if unavailable.
POINT_BUFFER_KM = [1, 2, 5]

## Optional Gas Flaring Mask

Use this when you want nighttime lights totals inside and outside buffers around gas flaring points. Your file should contain longitude and latitude columns.

In [ ]:
EXTRACT_GAS_FLARING_MASKS = False

GAS_FLARING_FILE = "gas_flaring/finaldata/gas_flaring_nga.csv"
GAS_FLARING_LONGITUDE_COL = "longitude"
GAS_FLARING_LATITUDE_COL = "latitude"
GAS_FLARING_BUFFER_KM = [5, 10]

## Paths and Token

Find the repository root, set project paths, and read your NASA Earthdata bearer token from `.config/ntl-training/secrets.env`.

In [ ]:
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / "spatial-blackmarble-training").exists():
            return path
    return start


def resolve_data_path(path):
    path = Path(path)
    return path if path.is_absolute() else DATA_DIR / path


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = DATA_DIR / NTL_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

secrets_path = REPO_ROOT / ".config" / "ntl-training" / "secrets.env"
secrets = dotenv_values(secrets_path)
blackmarble_token = secrets.get("BLACKMARBLE_TOKEN", "").strip()

if not blackmarble_token:
    blackmarble_token = getpass("Enter Black Marble token (input hidden): ").strip()

bm = BlackMarble(token=blackmarble_token)

## Load Administrative Boundaries

Load each administrative boundary file listed in `ADMIN_BOUNDARY_FILES`, repair geometries, and convert to `EPSG:4326`.

In [ ]:
admin_boundaries = {}

for admin_level, boundary_file in ADMIN_BOUNDARY_FILES.items():
    gdf = gpd.read_file(resolve_data_path(boundary_file))
    gdf["geometry"] = gdf.geometry.make_valid()

    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")

    if "date" in gdf.columns:
        gdf = gdf.drop(columns="date")

    admin_boundaries[admin_level] = gdf

adm0 = admin_boundaries[min(admin_boundaries)]
adm0_bbox = gpd.GeoDataFrame(geometry=[box(*adm0.total_bounds)], crs=adm0.crs)

admin_boundaries.keys()

## Load Optional Point Buffers

In [ ]:
point_buffers = {}
point_df = None

if EXTRACT_POINT_BUFFERS:
    point_path = resolve_data_path(POINT_FILE)
    point_df = pd.read_excel(point_path) if point_path.suffix.lower() in [".xls", ".xlsx"] else pd.read_csv(point_path)

    if POINT_ID_COL is None:
        point_df["point_id"] = range(1, len(point_df) + 1)
        POINT_ID_COL = "point_id"

    point_gdf_base = gpd.GeoDataFrame(
        point_df,
        geometry=gpd.points_from_xy(point_df[POINT_LONGITUDE_COL], point_df[POINT_LATITUDE_COL]),
        crs="EPSG:4326",
    )

    for km in POINT_BUFFER_KM:
        buffered = point_gdf_base.copy()
        buffered["geometry"] = buffered.to_crs(BUFFER_CRS).buffer(km * 1000).to_crs("EPSG:4326")
        point_buffers[km] = buffered

    print(f"Created point buffers for {len(point_gdf_base)} locations.")

## Load Optional Gas Flaring Buffers

In [ ]:
gas_flare_unions = {}

if EXTRACT_GAS_FLARING_MASKS:
    gas_path = resolve_data_path(GAS_FLARING_FILE)
    gas_flaring = pd.read_excel(gas_path) if gas_path.suffix.lower() in [".xls", ".xlsx"] else pd.read_csv(gas_path)

    gas_points = gpd.GeoDataFrame(
        gas_flaring,
        geometry=gpd.points_from_xy(gas_flaring[GAS_FLARING_LONGITUDE_COL], gas_flaring[GAS_FLARING_LATITUDE_COL]),
        crs="EPSG:4326",
    )

    for km in GAS_FLARING_BUFFER_KM:
        buffered = gas_points.copy()
        buffered["geometry"] = buffered.to_crs(BUFFER_CRS).buffer(km * 1000).to_crs("EPSG:4326")
        gas_flare_unions[km] = buffered.union_all()

    print(f"Created gas flaring masks for {GAS_FLARING_BUFFER_KM} km buffers.")

## Helper Functions

In [ ]:
def product_settings(product):
    if product == Product.VNP46A4:
        return "annual", "YS", END_DATE_ANNUAL
    if product == Product.VNP46A3:
        return "monthly", "MS", END_DATE_DAILY_MONTHLY
    if product == Product.VNP46A2:
        return "daily", "D", END_DATE_DAILY_MONTHLY
    raise ValueError(f"Unsupported product: {product}")


def csv_missing_dates(path, expected_dates):
    if not path.exists():
        return list(expected_dates)
    try:
        existing = pd.read_csv(path, usecols=["date"])
        existing_dates = set(pd.to_datetime(existing["date"], errors="coerce").dropna())
    except Exception:
        return list(expected_dates)
    return [d for d in expected_dates if pd.Timestamp(d) not in existing_dates]


def append_csv(path, new_rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        existing = pd.read_csv(path)
        combined = pd.concat([existing, new_rows], ignore_index=True, sort=False)
        cols = ["date"] + [c for c in combined.columns if c != "date"] if "date" in combined.columns else list(combined.columns)
        combined[cols].to_csv(path, index=False)
    else:
        new_rows.to_csv(path, index=False)


def make_mask_variants(gdf, mask_union):
    gdf = gdf.copy()
    gdf["_merge_id"] = range(len(gdf))

    gdf_in = gdf.copy()
    gdf_in["geometry"] = gdf_in.geometry.intersection(mask_union).make_valid()
    gdf_in = gdf_in[~gdf_in.is_empty & gdf_in.geometry.is_valid].reset_index(drop=True)

    gdf_out = gdf.copy()
    gdf_out["geometry"] = gdf_out.geometry.difference(mask_union).make_valid()
    gdf_out = gdf_out[~gdf_out.is_empty & gdf_out.geometry.is_valid].reset_index(drop=True)

    return gdf, gdf_in, gdf_out


def save_rasters(rasters_ds, raster_dir, product_name):
    for var in rasters_ds.data_vars:
        da = rasters_ds[var]
        time_dim = "time" if "time" in da.dims else da.dims[0]
        for t_idx in range(da.sizes[time_dim]):
            slice_da = da.isel({time_dim: t_idx})
            date_val = pd.Timestamp(da[time_dim].values[t_idx])
            out_file = raster_dir / f"{product_name}_{date_val.strftime('%Y%m%d')}.tif"
            slice_da.rio.to_raster(out_file)

## Download Rasters and Extract Statistics

Run this cell when your settings are ready. It writes output CSVs under `data/ntl_blackmarble/<COUNTRY_CODE>/aggregated/`.

In [ ]:
for product in PRODUCTS_TO_RUN:
    folder, freq, end_date = product_settings(product)
    product_name = product.name
    full_date_range = pd.date_range(START_DATE, end_date, freq=freq)

    raw_dir = OUTPUT_DIR / RAW_SUBDIR / folder
    raster_dir = OUTPUT_DIR / RAW_SUBDIR / "rasters" / folder
    aggregated_dir = OUTPUT_DIR / AGGREGATED_SUBDIR / folder
    raw_dir.mkdir(parents=True, exist_ok=True)
    raster_dir.mkdir(parents=True, exist_ok=True)
    aggregated_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting {product_name} for {COUNTRY_NAME} ({folder})")

    raster_files_exist = all(
        (raster_dir / f"{product_name}_{date.strftime('%Y%m%d')}.tif").exists()
        for date in full_date_range
    )

    if not raster_files_exist:
        def download_and_save(date):
            date_str = date.strftime("%Y%m%d")
            out_file = raster_dir / f"{product_name}_{date_str}.tif"
            if out_file.exists():
                return date_str, "exists"

            kwargs = {}
            if BLACKMARBLE_VARIABLE is not None:
                kwargs["variable"] = BLACKMARBLE_VARIABLE

            try:
                rasters_ds = raster.bm_raster(
                    adm0_bbox,
                    product,
                    pd.date_range(date, date, freq=freq),
                    token=blackmarble_token,
                    output_directory=str(raw_dir),
                    output_skip_if_exists=True,
                    **kwargs,
                )
                save_rasters(rasters_ds, raster_dir, product_name)
                return date_str, "downloaded"
            except ValueError:
                return date_str, "missing"
            except Exception as err:
                print(f"  Warning: unexpected error for {date_str}: {err}")
                return date_str, "error"

        max_workers = min(8, os.cpu_count() or 4)
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(download_and_save, date): date for date in full_date_range}
            for future in as_completed(futures):
                date_str, status = future.result()
                if status in ["downloaded", "missing", "error"]:
                    print(f"  {date_str}: {status}")

    for admin_level, gdf in admin_boundaries.items():
        out_path = aggregated_dir / f"ntl_{COUNTRY_CODE}_adm{admin_level}_{folder}.csv"
        missing_dates = csv_missing_dates(out_path, full_date_range)

        if not missing_dates:
            print(f"  Admin level {admin_level}: all dates present - skipping.")
            continue

        date_range = pd.DatetimeIndex(missing_dates)
        extract_kwargs = dict(
            product_id=product,
            date_range=date_range,
            token=blackmarble_token,
            output_directory=str(raw_dir),
            output_skip_if_exists=True,
        )

        gdf_with_id = gdf.copy()
        gdf_with_id["_merge_id"] = range(len(gdf_with_id))

        try:
            extracted = extract.bm_extract(gdf_with_id, **extract_kwargs)

            if EXTRACT_GAS_FLARING_MASKS:
                for km, mask_union in gas_flare_unions.items():
                    _, gdf_gf, gdf_nogf = make_mask_variants(gdf, mask_union)

                    for variant_gdf, prefix in [
                        (gdf_gf, f"ntl_gf_{km}km"),
                        (gdf_nogf, f"ntl_nogf_{km}km"),
                    ]:
                        col_name = f"{prefix}_sum"
                        if variant_gdf.empty:
                            extracted[col_name] = 0.0
                            continue

                        variant = extract.bm_extract(variant_gdf, **extract_kwargs)
                        variant = variant.rename(columns={"ntl_sum": col_name})
                        extracted = extracted.merge(
                            variant[["_merge_id", "date", col_name]],
                            on=["_merge_id", "date"],
                            how="left",
                        )
                        extracted[col_name] = extracted[col_name].fillna(0.0)

            new_rows = extracted.drop(columns=["geometry", "_merge_id", "_join_id"], errors="ignore")
            append_csv(out_path, new_rows)
            print(f"  Saved {len(new_rows)} new records to {out_path}")

        except ValueError as err:
            print(f"  bm_extract failed for ADM{admin_level} ({err}). Falling back to saved rasters.")
            records = []

            for tif_path in sorted(raster_dir.glob(f"{product_name}_*.tif")):
                try:
                    date_val = pd.Timestamp(tif_path.stem.split("_", 1)[1])
                except Exception:
                    continue
                if date_val not in set(date_range):
                    continue

                da = rioxarray.open_rasterio(tif_path, masked=True).squeeze()
                arr = da.values.astype("float64")
                transform = da.rio.transform()
                stats = zonal_stats(gdf_with_id, arr, affine=transform, stats=["sum"], nodata=np.nan, all_touched=True)

                for i, stat in enumerate(stats):
                    records.append({
                        "date": date_val,
                        "ntl_sum": stat["sum"] if stat["sum"] is not None else 0.0,
                        **gdf_with_id.drop(columns="geometry").iloc[i].to_dict(),
                    })

            if records:
                fallback_rows = pd.DataFrame(records).drop(columns=["_merge_id", "_join_id"], errors="ignore")
                append_csv(out_path, fallback_rows)
                print(f"  Saved {len(fallback_rows)} fallback records to {out_path}")

    if EXTRACT_POINT_BUFFERS:
        bc_out_path = aggregated_dir / f"ntl_{COUNTRY_CODE}_point_buffers_{folder}.csv"
        missing_dates = csv_missing_dates(bc_out_path, full_date_range)

        if missing_dates:
            point_records = []
            for date in pd.DatetimeIndex(missing_dates):
                tif_path = raster_dir / f"{product_name}_{date.strftime('%Y%m%d')}.tif"
                if not tif_path.exists():
                    continue

                for km, buffer_gdf in point_buffers.items():
                    stats = zonal_stats(buffer_gdf, str(tif_path), stats=["sum", "mean", "std"], all_touched=True)
                    for i, stat in enumerate(stats):
                        point = point_df.iloc[i]
                        record = {
                            "date": date,
                            "buffer_km": km,
                            "point_id": point[POINT_ID_COL],
                            "ntl_sum": stat.get("sum"),
                            "ntl_mean": stat.get("mean"),
                            "ntl_std": stat.get("std"),
                        }
                        if POINT_NAME_COL is not None and POINT_NAME_COL in point:
                            record["point_name"] = point[POINT_NAME_COL]
                        if POINT_GROUP_COL is not None and POINT_GROUP_COL in point:
                            record["point_group"] = point[POINT_GROUP_COL]
                        point_records.append(record)

            if point_records:
                point_rows = pd.DataFrame(point_records)
                append_csv(bc_out_path, point_rows)
                print(f"  Saved {len(point_rows)} point buffer records to {bc_out_path}")